# RAW to RGB — Part 3: White Balance and Edge-Directed Debayering

In this notebook we:
1. Apply **white balance** by sampling a neutral grey from the image
2. Implement **edge-directed (gradient-aware) debayering** to reduce zipper artifacts

The key idea of edge-directed debayering: instead of always averaging neighbours in all
four directions, we look at the local gradient and interpolate **along** the edge
(where the signal is smooth) rather than **across** it (where it changes rapidly).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)

data = np.load('21_RGB_BilinearDebayer.npz')
rgb                              = data['rgb']
bayer                            = data['img_black_subtracted_for_debayering']
black_noise_std                  = data['black_noise_std']

print(f'Loaded: rgb {rgb.shape},  bayer {bayer.shape}')

## White Balance

The camera's sensor responds differently in R, G, B to a neutral grey.
We correct this by **dividing each channel by the value it measures on a neutral patch**.

Sample a grey patch using the cursor in the image below (pick values from a white or
grey card in the scene), then enter them as `mean_white_R/G/B`.

> **Important:** Sample from the *un-gamma-corrected* image (`imshow(rgb)`) so that
> the values you read match the linear domain we are working in.

In [ ]:
# Display the raw (no gamma) image for sampling
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(np.clip(rgb, 0, 1))
ax.set_title('Linear RGB — use this image to pick white balance values')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Values sampled from a neutral grey / white patch in the image
mean_white_R = 0.115
mean_white_G = 0.280
mean_white_B = 0.155

wb = np.array([mean_white_R, mean_white_G, mean_white_B])

rgb_wb = rgb / wb   # broadcast over H x W

stops = 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(linear2sRGB(np.clip(rgb * 2**stops, 0, 1)))
axes[0].set_title('Without white balance')
axes[0].axis('off')

axes[1].imshow(linear2sRGB(np.clip(rgb_wb, 0, 1)))
axes[1].set_title('With white balance')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Edge-Directed Debayering

Bilinear debayering averages all four neighbours equally, which creates
**zipper artifacts** at sharp edges. The edge-directed approach detects the
local gradient direction and interpolates along the edge instead of across it.

### Green channel interpolation

For the green channel at a missing pixel, we compare the horizontal and vertical gradient:

- $\Delta_H = |G_{left} - G_{right}|$  
- $\Delta_V = |G_{above} - G_{below}|$

Then:
- if $\Delta_H > \Delta_V$: interpolate **vertically** (edge runs left–right)
- if $\Delta_V > \Delta_H$: interpolate **horizontally** (edge runs top–bottom)
- if equal: use the standard bilinear average

### R and B channels — chrominance trick

Instead of interpolating R and B directly, we interpolate the **chrominance** $R-G$ and $B-G$
using the regular bilinear kernels, then add back the (already well-interpolated) green.
This preserves colour accuracy at edges.

In [ ]:
def g_interpolation_edge_directed(block):
    """
    Edge-directed green interpolation for a 4×4 Bayer block.
    Input:  4×4 array of raw Bayer values
    Output: 2×2 array of interpolated green values
    
    Block layout (RGGB, rows/cols 1-indexed as in the lecture):
      row 1: G  R  G  R
      row 2: B  G  B  G
      row 3: G  R  G  R
      row 4: B  G  B  G
    After the row-crop in notebook 21 our pattern starts at:
      (0,0)=R  (0,1)=G  — so the 4x4 context window is:
      x[0,0] x[0,1] x[0,2] x[0,3]
      x[1,0] x[1,1] x[1,2] x[1,3]   <-- centre 2×2 we fill
      x[2,0] x[2,1] x[2,2] x[2,3]
      x[3,0] x[3,1] x[3,2] x[3,3]
    Green is known at x[1,2] (top-right) and x[2,1] (bottom-left).
    We estimate x[1,1] (upper-left) and x[2,2] (lower-right).
    """
    x = block  # 0-indexed

    # Gradient for upper-left missing green
    delta_H_ul = abs(x[1, 0] - x[1, 2])
    delta_V_ul = abs(x[0, 1] - x[2, 1])

    # Gradient for lower-right missing green
    delta_H_lr = abs(x[2, 1] - x[2, 3])
    delta_V_lr = abs(x[1, 2] - x[3, 2])

    y = np.full((2, 2), np.nan)
    y[0, 1] = x[1, 2]   # known green (top-right)
    y[1, 0] = x[2, 1]   # known green (bottom-left)

    # Upper-left
    if delta_H_ul > delta_V_ul:
        y[0, 0] = (x[0, 1] + x[2, 1]) / 2      # interpolate vertically
    elif delta_V_ul > delta_H_ul:
        y[0, 0] = (x[1, 0] + x[1, 2]) / 2      # interpolate horizontally
    else:
        y[0, 0] = (x[1, 0] + x[0, 1] + x[2, 1] + x[1, 2]) / 4  # bilinear

    # Lower-right
    if delta_H_lr > delta_V_lr:
        y[1, 1] = (x[1, 2] + x[3, 2]) / 2      # interpolate vertically
    elif delta_V_lr > delta_H_lr:
        y[1, 1] = (x[2, 1] + x[2, 3]) / 2      # interpolate horizontally
    else:
        y[1, 1] = (x[2, 1] + x[1, 2] + x[3, 2] + x[2, 3]) / 4  # bilinear

    return y


# Quick unit test — vertical edge: green should interpolate horizontally
test_block = np.array([[1, 0.9, 0.1, 0],
                        [1, 1.0, 0.2, 0],
                        [1, 0.8, 0.1, 0],
                        [1, 0.9, 0.0, 0]], dtype=float)
print('Edge-directed test result:')
print(g_interpolation_edge_directed(test_block))

In [ ]:
# Apply white balance to the raw Bayer plane before debayering
# (WB pattern follows the RGGB Bayer mosaic)
H, W = bayer.shape
wb_cfa = np.empty((H, W))
wb_cfa[0::2, 0::2] = mean_white_R
wb_cfa[0::2, 1::2] = mean_white_G
wb_cfa[1::2, 0::2] = mean_white_G
wb_cfa[1::2, 1::2] = mean_white_B

bayer_wb = bayer / wb_cfa

# ---- Edge-directed green interpolation (block-by-block) ----
# We process every non-overlapping 2×2 block using a 4×4 context window.
# For speed we vectorise using numpy stride tricks instead of Python loops.

pad = np.pad(bayer_wb, 1, mode='reflect')   # 1-pixel border for context

g_edge = np.zeros((H, W))

# Copy known green values first
g_edge[0::2, 1::2] = bayer_wb[0::2, 1::2]  # top-right green
g_edge[1::2, 0::2] = bayer_wb[1::2, 0::2]  # bottom-left green

# Fill missing green values (at R and B positions)
for row in range(0, H, 2):
    for col in range(0, W, 2):
        block = pad[row:row+4, col:col+4]
        result = g_interpolation_edge_directed(block)
        g_edge[row,   col  ] = result[0, 0]   # upper-left  (R position)
        r_end = min(row+1, H-1)
        c_end = min(col+1, W-1)
        if r_end < H and c_end < W:
            g_edge[r_end, c_end] = result[1, 1]   # lower-right (B position)

print('Green channel done.')

In [ ]:
# ---- R and B via chrominance interpolation ----
# Interpolate (R-G) and (B-G) chrominance using bilinear kernels,
# then recover R = (R-G) + G and B = (B-G) + G.


chroma_RG = bayer_wb - g_edge
chroma_BG = bayer_wb - g_edge

# Masks for R and B positions
mask_R = np.zeros((H, W), bool); mask_R[0::2, 0::2] = True
mask_B = np.zeros((H, W), bool); mask_B[1::2, 1::2] = True

# Bilinear kernel (same as notebook 21)
kernel_RB = np.array([[1, 2, 1],
                       [2, 4, 2],
                       [1, 2, 1]]) / 4.0

def np_convolve2d(image, kernel, mode='mirror'):
    """Pure NumPy 2D convolution — replaces scipy.ndimage.convolve."""
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    pad_mode = 'reflect' if mode == 'mirror' else 'constant'
    img_pad  = np.pad(image, ((ph, ph), (pw, pw)), mode=pad_mode)
    H, W     = image.shape
    out      = np.zeros((H, W))
    for i in range(kh):
        for j in range(kw):
            out += kernel[i, j] * img_pad[i:i+H, j:j+W]
    return out

def bilinear_chroma(chroma, mask, kernel):
    values = chroma * mask
    weight = np_convolve2d(mask.astype(float), kernel, mode='mirror')
    interp = np_convolve2d(values, kernel, mode='mirror') / np.maximum(weight, 1e-6)
    return np.where(mask, chroma, interp)

chroma_RG_interp = bilinear_chroma(chroma_RG, mask_R, kernel_RB)
chroma_BG_interp = bilinear_chroma(chroma_BG, mask_B, kernel_RB)

r_edge = chroma_RG_interp + g_edge
b_edge = chroma_BG_interp + g_edge

rgb_edge = np.stack([r_edge, g_edge, b_edge], axis=2)
print('Edge-directed debayering complete.')

## Compare: bilinear vs. edge-directed

In [ ]:
vo, ho = 1220, 1050
size   = 200

crop_bilinear = rgb[vo*2:(vo+size)*2, ho*2:(ho+size)*2, :] / wb
crop_edge     = rgb_edge[vo*2:(vo+size)*2, ho*2:(ho+size)*2, :]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(linear2sRGB(np.clip(crop_bilinear, 0, 1)))
axes[0].set_title('Bilinear interpolation')
axes[0].axis('off')

axes[1].imshow(linear2sRGB(np.clip(crop_edge, 0, 1)))
axes[1].set_title('Edge-directed interpolation')
axes[1].axis('off')

plt.suptitle('Edge sharpness comparison — zipper artifacts reduced')
plt.tight_layout()
plt.show()

## Save result

In [ ]:
rgb_final = rgb_edge   # use edge-directed result going forward

np.savez('22_RGB_EdgeDirectedDebayer.npz',
         rgb=rgb_final,
         black_noise_std=black_noise_std)

print('Saved: 22_RGB_EdgeDirectedDebayer.npz')